# Train a Custom YOLOv8 Toy-Car Detector

Fine-tunes `yolov8n.pt` on the dataset produced by `prepare_toy_dataset.py`.
The result (`best.pt`) is copied to the repo root as `toy_cars.pt`.

This notebook mirrors `dataset_tools/train_toy_model.py`.

> **Note:** run this notebook from the **repo root** so the relative paths resolve.
> The first code cell handles that automatically.

In [ ]:
# Make sure we're at the repo root (adjust if your repo lives elsewhere)
import os

# If this notebook is opened from dataset_tools/, hop up one level to the repo root.
if os.path.basename(os.getcwd()) == "dataset_tools":
    os.chdir("..")

print("Working directory:", os.getcwd())

In [ ]:
import os
import shutil

# --- Configuration (the argparse defaults from train_toy_model.py) ---
DATA_YAML = os.path.join("data", "datasets", "toy_cars", "data.yaml")
OUTPUT_WEIGHTS = "toy_cars.pt"

DATA = DATA_YAML        # path to data.yaml
BASE = "yolov8n.pt"     # base weights to fine-tune
EPOCHS = 10             # number of training epochs
IMGSZ = 640             # training image size
BATCH = 16              # batch size
NAME = "toy_cars"       # run name under runs/detect/

In [ ]:
# Sanity check: the dataset must exist first (created by prepare_toy_dataset.py)
if not os.path.isfile(DATA):
    raise FileNotFoundError(f"{DATA} not found. Run prepare_toy_dataset.py first.")

print(f"[OK] Found dataset config: {DATA}")

In [ ]:
from ultralytics import YOLO

print(f"[TRAIN] base={BASE} data={DATA} epochs={EPOCHS} imgsz={IMGSZ}")
model = YOLO(BASE)
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name=NAME,
)

In [ ]:
# Locate best.pt from the run and copy it to the repo root for easy use
save_dir = getattr(results, "save_dir", None) or os.path.join("runs", "detect", NAME)
best = os.path.join(str(save_dir), "weights", "best.pt")

if os.path.isfile(best):
    shutil.copy2(best, OUTPUT_WEIGHTS)
    print(f"\n[DONE] Best weights: {best}")
    print(f"[DONE] Copied to:    {OUTPUT_WEIGHTS}")
    print("\nNow wire it into config.py:")
    print(f'    YOLO_MODEL = "{OUTPUT_WEIGHTS}"')
    print("    VEHICLE_CLASSES = [0]   # custom model has a single class: 0 = car")
else:
    print(f"[WARN] Could not find best.pt under {save_dir}. Check the run output above.")

## Quick Prediction / Visualization

Run the trained model on a sample image and display the detections inline.
This loads `toy_cars.pt` (the weights exported above) and draws boxes for any detected cars.

Set `SAMPLE_IMAGE` to any image path. By default it grabs the first validation image.

In [ ]:
import glob
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Pick a sample image: use an explicit path, or fall back to the first val image.
SAMPLE_IMAGE = None  # e.g. "data/datasets/toy_cars/images/val/frame_0001.jpg"

if SAMPLE_IMAGE is None:
    val_dir = os.path.join("data", "datasets", "toy_cars", "images", "val")
    candidates = sorted(
        glob.glob(os.path.join(val_dir, "*.jpg"))
        + glob.glob(os.path.join(val_dir, "*.png"))
    )
    if not candidates:
        raise FileNotFoundError(f"No images found in {val_dir}")
    SAMPLE_IMAGE = candidates[0]

print(f"Predicting on: {SAMPLE_IMAGE}")

# Load the freshly trained weights and run inference
pred_model = YOLO(OUTPUT_WEIGHTS)
pred_results = pred_model.predict(SAMPLE_IMAGE, conf=0.15, verbose=False)

# Report what was found
result = pred_results[0]
print(f"Detections: {len(result.boxes)}")
for box in result.boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    print(f"  class={cls_id} ({result.names[cls_id]})  conf={conf:.2f}")

# result.plot() returns a BGR annotated image; flip to RGB for matplotlib
annotated = result.plot()[:, :, ::-1]
plt.figure(figsize=(10, 7))
plt.imshow(annotated)
plt.axis("off")
plt.title(os.path.basename(SAMPLE_IMAGE))
plt.show()